# Fraud Detection - Feature Engineering

This notebook covers:
- Feature Store setup with online-enabled FeatureViews
- Customer profile features (static, daily refresh)
- **Online** aggregated transaction features (current 7-day window per customer, 1-minute refresh) for real-time serving
- **Point-in-time** aggregated transaction features (per transaction, only prior transactions) for training
- Derived risk features computed inline
- Training dataset generation via Feature Store spine join

All feature definitions (names, aggregation logic, derived features) come from `source/utils.py`, the single source of truth shared with training (`source/train.py`) and serving (`simulator/`).

### Import Libraries

Import Feature Store APIs for managing ML features and Snowpark for defining feature transformations.

In [ ]:
import sys
sys.path.append('../source')

from snowflake.snowpark import Session
from snowflake.snowpark.types import *
from snowflake.snowpark import functions as F
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, OnlineConfig
from snowflake.ml.feature_store.feature_view import FeatureViewStatus
import pandas as pd
import numpy as np

from utils import (
    AGG_WINDOW_DAYS,
    BASELINE_SCORE_COL,
    FEATURE_COLS,
    ID_COL,
    LABEL_COL,
    TIMESTAMP_COL,
    add_derived_features_snowpark,
    build_current_agg_features_snowpark,
    build_point_in_time_agg_features_snowpark,
)

AGG_FV_VERSION = "V5"       # online, per-customer, current window (serving)
PIT_AGG_FV_VERSION = "V1"   # offline, per-transaction, point-in-time (training)

### Initialize Session

In [ ]:
import os
import snowflake.connector
from snowflake.snowpark import Session

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    conn = snowflake.connector.connect(
        connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME")
    )
    session = Session.builder.configs({"connection": conn}).create()

session.sql("USE ROLE ACCOUNTADMIN").collect()
print(f"Role set to: {session.get_current_role()}")

### Set Database Context

Establish the working context for all subsequent operations. Using consistent database/schema across notebooks ensures all artifacts are created in the right location.

In [ ]:
DATABASE = "DEMO_FRAUD_DETECTION_DB"
SCHEMA = "ML_PIPELINE"
FEATURE_STORE_SCHEMA = "FEATURE_STORE"
WAREHOUSE = "ML_WH"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {WAREHOUSE}").collect()

## Initialize Feature Store

In [ ]:
fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=FEATURE_STORE_SCHEMA,
    default_warehouse=WAREHOUSE
)
print(f"Connected to Feature Store: {DATABASE}.{FEATURE_STORE_SCHEMA}")

## Create Customer Profile Feature View

### Examine Customer Profile Data

These **static features** (account age, credit score, fraud history) change infrequently but are critical for fraud risk assessment.

In [ ]:
customer_profiles_df = session.table(f"{DATABASE}.{SCHEMA}.CUSTOMER_PROFILES")
print(f"Customer profiles columns: {customer_profiles_df.columns}")
customer_profiles_df.show(5)

### Register Customer Profile Feature View

Create a Feature View for customer profile data:
- **refresh_freq='1 day'**: Profiles update daily (static data)
- **entities=[customer_entity]**: Join key is CUSTOMER_ID
- **version='V1'**: Enables tracking feature evolution

In [ ]:
customer_entity = fs.get_entity("CUSTOMER")

online_config = OnlineConfig(enable=True)

customer_profile_fv = FeatureView(
    name="CUSTOMER_PROFILE_FEATURES",
    entities=[customer_entity],
    feature_df=customer_profiles_df,
    refresh_freq="1 day",
    online_config=online_config,
    desc="Customer profile features for fraud detection - online enabled for real-time serving"
)

customer_profile_fv = fs.register_feature_view(
    feature_view=customer_profile_fv,
    version="V1",
    overwrite=True
)
print(f"Registered: {customer_profile_fv.name} (online enabled)")

## Create Aggregated Transaction Features

### Build Aggregated Transaction Features

We define the feature transformations using a **Snowpark DataFrame** — Snowflake's lazy, server-side DataFrame API. Snowpark DataFrames build a query plan that gets translated to SQL and executed on warehouse.

Two views of the same aggregates (identical column names, built from the same expressions in `source/utils.py`):

| Feature View | Entity | Window | Used for |
|---|---|---|---|
| `CUSTOMER_AGG_FEATURES` | CUSTOMER | last `AGG_WINDOW_DAYS` days **as of now** | Online serving (real-time lookups) |
| `TRANSACTION_PIT_AGG_FEATURES` | TRANSACTION | `[ts - AGG_WINDOW_DAYS, ts)` — strictly **before** each transaction | Training (point-in-time correct) |

Training on per-customer lifetime/"as of now" aggregates leaks future behaviour (including the fraudulent transaction itself) into the features. The point-in-time view uses a self-join restricted to earlier transactions, so every training row only sees what was knowable when the transaction happened — the same thing the online view returns at request time.

Customers without prior history get `HAS_HISTORY = 0`, zero counts/sums and NULL means/std/min/max/ratios (XGBoost handles missing values natively) instead of silent zero-filling.

**Note:** For heavier feature engineering that requires custom Python/ML logic at scale you can use **Container Runtime + Ray Data** on SPCS compute pools instead.

In [ ]:
transactions_df = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")

# Online / serving: one row per customer, current window
agg_features_df = build_current_agg_features_snowpark(transactions_df, AGG_WINDOW_DAYS)

# Offline / training: one row per transaction, prior transactions only
pit_agg_features_df = build_point_in_time_agg_features_snowpark(transactions_df, AGG_WINDOW_DAYS)

print(f"Online aggregate columns ({AGG_WINDOW_DAYS}-day window): {agg_features_df.columns}")
print(f"Point-in-time aggregate columns: {pit_agg_features_df.columns}")

### Register Aggregated Features as Feature Views

Pass the transformation DataFrames directly to `FeatureView` — the Feature Store creates the backing Dynamic Tables automatically:
- **CUSTOMER_AGG_FEATURES** (`refresh_freq='1 minute'`, online enabled): new transactions update serving aggregates within 1 minute
- **TRANSACTION_PIT_AGG_FEATURES** (`refresh_freq='1 day'`, offline): per-transaction point-in-time aggregates for training datasets

In [ ]:
agg_features_fv = FeatureView(
    name="CUSTOMER_AGG_FEATURES",
    entities=[customer_entity],
    feature_df=agg_features_df,
    refresh_freq="1 minute",
    online_config=online_config,
    desc=f"Current {AGG_WINDOW_DAYS}-day customer aggregates (D{AGG_WINDOW_DAYS}_ prefixed, HAS_HISTORY) - online enabled for real-time serving"
)
agg_features_fv = fs.register_feature_view(
    feature_view=agg_features_fv,
    version=AGG_FV_VERSION,
    block=True,
    overwrite=True
)
print(f"Registered: {agg_features_fv.name} (version {AGG_FV_VERSION}, online enabled)")

transaction_entity = fs.get_entity("TRANSACTION")
pit_agg_features_fv = FeatureView(
    name="TRANSACTION_PIT_AGG_FEATURES",
    entities=[transaction_entity],
    feature_df=pit_agg_features_df,
    refresh_freq="1 day",
    desc=f"Point-in-time {AGG_WINDOW_DAYS}-day aggregates over transactions strictly before each transaction (training)"
)
pit_agg_features_fv = fs.register_feature_view(
    feature_view=pit_agg_features_fv,
    version=PIT_AGG_FV_VERSION,
    block=True,
    overwrite=True
)
print(f"Registered: {pit_agg_features_fv.name} (version {PIT_AGG_FV_VERSION}, offline)")

## Generate Training Dataset

### Create Spine DataFrame

The **spine** defines which entity instances to generate features for:
- Contains primary keys (TRANSACTION_ID, CUSTOMER_ID)
- One row per training example
- Feature Store joins features to this spine

**Why a spine?** It ensures we generate features only for the exact entities/timestamps we need, avoiding wasted computation.

In [ ]:
transactions_df = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")
spine_df = transactions_df.select("TRANSACTION_ID", "CUSTOMER_ID")

print(f"Spine dataframe rows: {spine_df.count():,}")

### Generate Dataset from Feature Store

Use Feature Store's `generate_dataset()` to automatically join features to the spine:
- Customer profile features join on `CUSTOMER_ID`
- Point-in-time aggregates join on `TRANSACTION_ID` (each row already only reflects prior transactions)
- Creates a versioned, reproducible dataset

**Key benefit**: Training and serving use identical feature definitions (`source/utils.py`), preventing training-serving skew.

In [ ]:
training_data = fs.generate_dataset(
    name="FRAUD_TRAINING_DATASET",
    spine_df=spine_df,
    features=[
        fs.get_feature_view("CUSTOMER_PROFILE_FEATURES", "V1"),
        fs.get_feature_view("TRANSACTION_PIT_AGG_FEATURES", PIT_AGG_FV_VERSION),
    ],
    spine_timestamp_col=None,
    include_feature_view_timestamp_col=False,
    desc=f"Training dataset for fraud detection model (point-in-time {AGG_WINDOW_DAYS}-day aggregates)",
    output_type="dataset"
)

print(f"Generated training dataset")

### Combine All Features into Final Training Set

Join Feature Store data with:
1. **Transaction attributes**: Amount, time, card presence, etc. plus `TRANSACTION_TIMESTAMP` (needed for the time-based train/test split)
2. **Labels**: IS_FRAUD (target variable)
3. **Derived features**: `add_derived_features_snowpark()` from `source/utils.py` (risk ratios, anomaly flags, minutes since last transaction)

`RISK_SCORE` is kept as a **baseline comparison column only** — it is a hand-built heuristic that partially encodes the label-generation rules, so it is excluded from `FEATURE_COLS` and never used as a model input.

In [ ]:
training_df = training_data.read.to_snowpark_dataframe()
transactions_full = session.table(f"{DATABASE}.{SCHEMA}.TRANSACTIONS")

final_training_df = add_derived_features_snowpark(
    training_df.join(
        transactions_full.select(
            ID_COL,
            TIMESTAMP_COL,
            "TRANSACTION_AMOUNT",
            "TRANSACTION_HOUR",
            "TRANSACTION_DAY_OF_WEEK",
            "IS_WEEKEND",
            "CARD_PRESENT",
            "IS_INTERNATIONAL",
            LABEL_COL,
        ),
        on=ID_COL,
        how="inner",
    ),
    AGG_WINDOW_DAYS,
).select(ID_COL, "CUSTOMER_ID", TIMESTAMP_COL, *FEATURE_COLS, BASELINE_SCORE_COL, LABEL_COL)

print(f"Model features ({len(FEATURE_COLS)}): {FEATURE_COLS}")
print(f"Total rows: {final_training_df.count():,}")

### Save Training Data

Persist the final training dataset as a Snowflake table. This enables:
- Reproducible model training across notebook sessions
- Data versioning and lineage tracking
- Efficient distributed training with Snowflake ML

In [ ]:
final_training_df.write.mode("overwrite").save_as_table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")
print(f"Saved training data to {DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")

## Verify Feature Views

### Summary and Validation

Verify all Feature Views are registered and training data statistics:
- Confirm feature counts and data volumes
- Check class distribution (expect ~2% fraud)
- List all registered Feature Views for reference

In [ ]:
print("\n=== Feature Store Summary ===")
print("\nRegistered Feature Views:")
for fv in fs.list_feature_views().collect():
    print(f"  - {fv['NAME']} (version: {fv['VERSION']})")

print(f"\nTraining Data Statistics:")
train_df = session.table(f"{DATABASE}.{SCHEMA}.FRAUD_TRAINING_DATA")
print(f"  - Total samples: {train_df.count():,}")
print(f"  - Model features: {len(FEATURE_COLS)} (+ {BASELINE_SCORE_COL} baseline, excluded from training)")
print(f"  - Rows without prior history: {train_df.filter(F.col('HAS_HISTORY') == 0).count():,}")
fraud_stats = train_df.group_by("IS_FRAUD").count().collect()
for row in fraud_stats:
    label = "Fraud" if row["IS_FRAUD"] == 1 else "Legitimate"
    print(f"  - {label}: {row['COUNT']:,}")